# UKFによる非線形システムの推定

- 状態推定
- 外乱推定
- パラメータ推定

## 概要

UKFは非線形システムの推定に用いられるアルゴリズムである。



### 解きたい問題

UKFは不確かな状態$x$を非線形な状態方程式$f$に通した後、通過した状態の平均と分散により、その状態の推定値を求める方法である。

例えば、1次元の状態$x_k$を考え、それが以下の平均$\mu_k$,分散$P_k$の正規分布に従う確率変数だとする。

$$
x_k \sim \mathcal{N}(\mu_k, P_k)
$$

この状態$x_k$を、非線形な状態方程式$f$を通過させると、次の状態$x_{k+1}$は以下のようになる。

$$
x_{k+1} = f(x_k)
$$

この状態$x_{k+1}$は確率変数である。この状態の推定値として$\mu_{k+1}$を設定する。 

$$
\mu_{k+1} = E[x_{k+1}] = E[f(x_k)]
$$

また、この推定がどれくらい不確かなのかを求めるために、分散$P_{k+1}$も求める。

$$
P_{k+1} = E[(f(x_k) - \mu_{k+1})^2]
$$

$f(x_k)$が以下のように線形である場合、以下のように平均と分散を計算できる。

$$
x_{k+1} = f(x_k) = ax_k + b
$$

- 平均

    $$
    \mu_{{k+1}} = E[f(x_k)] = E[a x_k + b] = aE[x_k] + b = a \mu_k + b
    $$

    これより、平均値を状態方程式へ入れれば、次の平均値を得られる。

    $$
    \mu_{{k+1}} = a \mu_k + b \Rightarrow E[f(x)] = f(E[x])
    $$


- 分散

$$
\begin{aligned}
P_{k+1} &= E[(f(x_k)- \mu_{k+1})^2] \\
&= E[((ax_k + b) - (a \mu_k + b))^2] \\
&= E[(ax_k - a\mu_k)^2] \\
&= a^2E[(x_k-\mu_k)^2] \\
&= a^2 P_k
\end{aligned}
$$


しかし、例えば次のように状態方程式が非線形の場合

$$
x_{k+1} = f(x_k) = x_k^2 , \quad x_k \sim \mathcal{N}(0,1)
$$

現在の平均値$\mu_k$はゼロとなる。

$$
\mu_k = E[x_k] = 0
$$

線形状態方程式の関係性と同じく、「現状態の平均値を状態方程式へ入れると、次状態の推定値が得られる」と考えると、次のようになる。

$$
f(E[x]) = f(\mu_k) = f(0) = 0
$$

しかし実際に次の状態の平均は次のようになる。

$$
\mu_{k+1} = E[f(x)] = E[x_k^2] = E[(x_k - \mu_k)^2] = 1
$$

よって、非線形方程式では以下が成り立たない。

$$
f(E[x]) = 0 \neq E[f(x)] = 1
$$

これは元の状態が平均$0$の周りに

$$
-1 , \quad 0 , \quad 1
$$

のようにばらついていたとすると、$f(x)=x^2$を通すことによって、以下のように正側へ変換される。

$$
1 , \quad 0 , \quad 1
$$

よって、変換後の平均値は0ではなく、正側へ移動する。

よって、平均値$\mu_k$だけでなく、分布を代表するいくつかの点も$f$を通過させてデータの平均$\mu_{k+1}$と分散$P_{k+1}$を近似する。


### シグマポイント

#### 1次元の場合

##### 目的

現在の状態を以下とする。

$$
x \sim \mathcal{N}(\mu, P)
$$

この分布全体を非線形関数$f(x)$に通すことは大変なので、少数の代表点でこの分布を置き換えることを考える。

例えば次の3点とする。この3点を適当に選ぶのではなく、元の状態の平均$\mu$と分散$P$も再現できることが必要である。これをシグマポイントと呼ぶ。

$$
\chi_0 , \chi_1, \chi_2
$$

##### シグマポイントの設定

次のようにシグマポイントを平均とその左右対称に設定する。

$$
\chi_0 = \mu, \quad \chi_1 = \mu + c , \quad \chi_2 = \mu -c
$$

それぞれのポイントに対する重みを以下とする。

$$
W_0, W_1, W_2
$$

平均に対して左右対称であるため、以下のように設定する。

$$
W_1 = W_2 = W
$$

また、この3点で元の分布を表現するため、重みの合計が1である必要がある。

$$
W_0 + 2W = 1
$$

##### シグマポイントのよる平均

シグマポイントから計算した平均を$\hat{\mu}$とすると、

$$
\begin{aligned}
\hat{\mu} &= W_0 \chi_0 + W \chi_1 + W \chi_2 \\
&= W_0 \mu + W (\mu + c) + W (\mu - c) \\
&= (W_0 + 2W)\mu \\
&= \mu
\end{aligned}
$$

したがって、 $\hat{\mu} = \mu$となり、シグマポイントにより、元の分布の平均値を再現できる。

##### シグマポイントによる分散

シグマポイントから計算した分散を$\hat{P}$とすると、

$$
\begin{aligned}
\hat{P} &= \sum_i W_i(\chi_i - \mu)^2 \\
&= W_0 (\mu - \mu)^2 + W(\mu+c - \mu)^2 + W(\mu - c - \mu)^2 \\
&= Wc^2 + Wc^2 \\
&= 2Wc^2
\end{aligned}
$$

これを元の分散と一致させたいので、次のような係数$c$が必要である。

$$
2Wc^2 = P \quad \Rightarrow \quad c = \sqrt{\frac{P}{2W}}
$$

これより、シグマポイントは$\sqrt{P}$に比例した距離となり、このシグマポイントで元の分布の分散を再現することが出来る。

##### シグマポイントの例

仮に$W_0$と以下のように決める。

$$
W_0 = \frac{1}{2}
$$

重みの総和を1とするので、$W$は次のように決まる。

$$
W_0 + 2W = 1 \rightarrow W = \frac{1}{4}
$$

そして、シグマポイントとの係数$c$は次のようになる。

$$
c = \sqrt{\frac{P}{2W}} = \sqrt{2P}
$$

したがって、シグマポイントは次のように決まる。

$$
\chi_0 = \mu, \quad \chi_1 = \mu + \sqrt{2P} , \quad \chi_2 = \mu - \sqrt{2P}
$$


##### シグマポイントを非線形関数へ通す

状態$x$を前節と同じく以下とする。

$$
x \sim \mathcal{N}(0, 1)
$$

これによりシグマポイントは次のようになる。

$$
\chi_0 = 0, \quad \chi_1 = \sqrt{2} , \quad \chi_2 = -\sqrt{2}
$$

これを以下の非線形関数に通過させた後の平均と分散を求める。

$$
f(x) = x^2
$$

変換後のシグマポイントを$Y_i$とすると、

$$
Y_i = f(\chi_i)
$$

より、以下となる。

$$
Y_0 = 0, \quad Y_1 = 2, \quad Y_2 = 2
$$

変換後の平均を求めると"1"となる。

$$
\begin{aligned}
\mu_y &= \sum_i W_i Y_i \\
&= \frac{1}{2}(0) + \frac{1}{4}(2) + \frac{1}{4}(2) \\
&= 1
\end{aligned}
$$

これにより、確率変数である$x$の分布点をいくつも$f$に通さずとも、本当の期待値である、

$$
E[f(x)] = E[x^2] =1
$$

と一致する。

分散は以下となり、元の状態の分散と一致する。

$$
\begin{aligned}
P_y &= \sum_i W_i (Y_i - \mu_y)^2 \\
&= \frac{1}{2}(0 - 1)^2 + \frac{1}{4}(2 - 1)^2 + \frac{1}{4}(2-1)^2 \\
&= \frac{1}{2} + \frac{1}{4} + \frac{1}{4} \\
&= 1
\end{aligned}
$$